# 05 · Cheap B Pre-test

B 不是 A。A 赌主 pair 的方向；B 赌**同一曲线内部**哪个相邻 spread 的 carry 高、哪个低。

这个 pre-test 是预先注册的最小版本：

- 每个商品、每天，算 `M1-M2`、`M2-M3`、`M3-M4` 的年化 carry；
- long carry 最高的相邻 spread，short carry 最低的；
- 4 条腿（共享腿会净掉）；
- 同一 engine、K=5、weekly、5 bps。

**Decision rule（跑之前定好）：** full-sample `gross Sharpe < 0.2` 或 `net Sharpe <= 0` → 跳过 B。

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


In [2]:
import pandas as pd
from IPython.display import display

from src import b_pre_test as b


## 1. Build targets and run

`run_b_pretest` 建 K=5 主 pair、B 腿级 target、per-contract mark state，
然后跑 full 和 test。


In [3]:
rank = pd.read_parquet("data/processed/rank.parquet")
out = b.run_b_pretest(rank)
print("target rows:", out.targets.shape[0])
print(out.targets["leg"].value_counts().to_string())
sample = out.targets.loc[
    (out.targets["root"] == "CL") & (out.targets["date"] == "2021-10-01")
]
display(sample)


target rows: 872652
leg
extra    400512
far      236070
near     236070


,date,root,commodity,sector,group_id,contract,leg,units,target
763116,2021-10-01,CL,WTI Crude Oil,Energy,CL_CLX21_CLZ21,CLF22,extra,1.0,1.0
763117,2021-10-01,CL,WTI Crude Oil,Energy,CL_CLX21_CLZ21,CLG22,extra,-1.0,1.0
763118,2021-10-01,CL,WTI Crude Oil,Energy,CL_CLX21_CLZ21,CLZ21,far,1.0,1.0
763119,2021-10-01,CL,WTI Crude Oil,Energy,CL_CLX21_CLZ21,CLX21,near,-1.0,1.0


## 2. Result

`gross` 是没扣成本的 B spread return；`net` 扣 5 bps/leg。


In [4]:
display(out.summary)
out.summary.to_csv("output/b_pretest_summary.csv", index=False)
skip = b.should_skip_b(out.summary)
print("skip B?", skip)


,start,end,n_days,years,ann_return_gross,ann_return_net,ann_vol_gross,ann_vol_net,sharpe_gross,sharpe_net,max_drawdown_net,hit_rate_net,avg_N_t,avg_net_exposure,n_trades,turnover,period
0,1986-01-02,2026-10-13,10507,41.694444,-0.011418,-0.028372,0.009447,0.009683,-0.076280,-0.186925,-0.701065,0.405979,22.864852,0.004550,100117,17.371859,full
1,1986-01-02,2015-12-31,7721,30.638889,-0.011041,-0.027969,0.008294,0.008582,-0.084063,-0.207955,-0.582737,0.406270,22.636187,0.004573,72201,17.614855,train
2,2016-01-04,2026-10-13,2786,11.055556,-0.012530,-0.029555,0.012078,0.012226,-0.065380,-0.154183,-0.285072,0.404822,23.454056,0.004482,27930,17.006490,test


skip B? True


**Zero-cost check:** `ann_return_gross` / `sharpe_gross` 就是
**不考虑 transaction cost（0 bps）** 的结果。Full sample gross ann. −1.14%，
gross Sharpe −0.08；train 和 test 也分别为 −1.10% / −1.25%。
B 在成本前已经为负，5 bps 的 net −2.84% / −0.19 只是更差。

## 3. 结论

这个 cheap B pre-test **失败**：

- Full sample gross ann. 约 −1.14%，gross Sharpe 约 −0.08；
- Net ann. 约 −2.84%，net Sharpe 约 −0.19；
- Train 和 test 也都是负的；
- Gross 已经负，所以不是成本造成的。

按预注册 decision rule：**skip B**。

**Caveat：** 这是“相邻 carry rank”版本，不是 residual/fitted-curve 版本。
后者是另一个假设，理论上可能不同；但它需要拟合 4 个点、过拟合风险更高，
而且 A 和这个 pre-test 都没有给出 carry 家族有 edge 的证据。
如果以后还有时间，residual B 应该作为**独立的预注册 idea**，不是现在的 rescue。


## 4. 下一步

- B 以 negative pre-test 收尾，不做 shallow B。
- C 已经在另一个 branch 上完成 baseline；下一步是 C 的 robustness。
- A 的 negative diagnosis 在 `04_a_diagnose.ipynb`（已合入 main）。
- 不要用 test set 调参。